In [1]:
# import packages
import ee
import geemap

In [2]:
# authenticate the EE api
ee.Authenticate()

True

In [3]:
# initialize the EE api
ee.Initialize(project='y2y-climate-benefits')

## Define GEE datasets

In [5]:
# define EE datasets
# carbon stocks
agb_mean_act = ee.Image('projects/y2y-climate-benefits/assets/inputs/tcd_agbc_mean_act')
agb_mean_prim = ee.Image('projects/y2y-climate-benefits/assets/inputs/tcd_agbc_mean_prim')
bgb_mean_act = ee.Image('projects/y2y-climate-benefits/assets/inputs/tcd_bgbc_mean_act')
bgb_mean_prim = ee.Image('projects/y2y-climate-benefits/assets/inputs/tcd_bgbc_mean_prim')
soc_mean_act = ee.Image('projects/y2y-climate-benefits/assets/inputs/tcd_soc_mean_act')
soc_mean_prim = ee.Image('projects/y2y-climate-benefits/assets/inputs/tcd_soc_mean_prim')

# vector
y2y = ee.FeatureCollection("projects/y2y-climate-benefits/assets/inputs/y2y")

# area
# compute per-pixel area in ha
pixel_area_ha = ee.Image.pixelArea().divide(10000)

## Calculate Zonal Statistics

In [7]:
# Calculate actual and pristine carbon stocks
agb_act = (
    agb_mean_act
    .multiply(pixel_area_ha)
    .rename(['agb_act'])
    .reduceRegion( # calc sum
        reducer=ee.Reducer.sum(),
        geometry=y2y.geometry(),
        scale=agb_mean_act.projection().nominalScale(),
        maxPixels=1e20
    )
)

agb_prim = (
    agb_mean_prim
    .multiply(pixel_area_ha)
    .rename(['agb_prim'])
    .reduceRegion( # calc sum
        reducer=ee.Reducer.sum(),
        geometry=y2y.geometry(),
        scale=agb_mean_prim.projection().nominalScale(),
        maxPixels=1e20
    )
)

bgb_act = (
    bgb_mean_act
    .multiply(pixel_area_ha)
    .rename(['bgb_act'])
    .reduceRegion( # calc sum
        reducer=ee.Reducer.sum(),
        geometry=y2y.geometry(),
        scale=bgb_mean_act.projection().nominalScale(),
        maxPixels=1e20
    )
)

bgb_prim = (
    bgb_mean_prim
    .multiply(pixel_area_ha)
    .rename(['bgb_prim'])
    .reduceRegion( # calc sum
        reducer=ee.Reducer.sum(),
        geometry=y2y.geometry(),
        scale=bgb_mean_prim.projection().nominalScale(),
        maxPixels=1e20
    )
)

soc_act = (
    soc_mean_act
    .multiply(pixel_area_ha)
    .rename(['soc_act'])
    .reduceRegion( # calc sum
        reducer=ee.Reducer.sum(),
        geometry=y2y.geometry(),
        scale=soc_mean_act.projection().nominalScale(),
        maxPixels=1e20
    )
)

soc_prim = (
    soc_mean_prim
    .multiply(pixel_area_ha)
    .rename(['soc_prim'])
    .reduceRegion( # calc sum
        reducer=ee.Reducer.sum(),
        geometry=y2y.geometry(),
        scale=soc_mean_prim.projection().nominalScale(),
        maxPixels=1e20
    )
)

# export to drive
ee.batch.Export.table.toDrive(
    collection=ee.FeatureCollection([
        ee.Feature(None, {
            "agb_act": agb_act,
            "agb_prim": agb_prim,
            "bgb_act": bgb_act,
            "bgb_prim": bgb_prim,
            "soc_act": soc_act,
            "soc_prim": soc_prim
            })]),
    description="carbon_deficit_stats_y2y",
    folder="",
    fileFormat="CSV"
).start()